# 01 · Data Collection

Linus Media Group — channel network and owned storefront.

All collection happens inside this notebook. No shell required.

## Sources and methods

| Source | Method | What it gives us |
|---|---|---|
| All videos, 4 channels | **YouTube Data API v3** | Complete catalog, **exact `publishedAt`**, exact view counts, duration |
| Shorts classification | `yt-dlp` on the `/shorts` tab | YouTube's own Shorts labelling (IDs only) |
| LTT Store | Shopify `products.json` | Products, variant SKUs, prices, stock status |

**Why the API instead of `yt-dlp` for the video catalog.**
`yt-dlp`'s flat-playlist mode returns *approximate* publish dates — for anything older than a
year only the year survives, the month and day collapse to the scrape date — and view counts
are rounded to three significant figures. It also gets rate-limited a few thousand items deep
on the main channel. The API has neither problem. Cost is roughly 450 quota units against a
free daily allowance of 10,000.

**Why Shorts classification still needs `yt-dlp`.**
The API's uploads playlist returns long-form videos and Shorts interleaved with no field that
distinguishes them. Guessing from duration is unreliable, so we take the ID set from the
`/shorts` tab as ground truth for what YouTube itself considers a Short.

---
Setup: `pip install google-api-python-client python-dotenv yt-dlp pandas requests`

## 0. Environment setup

Run this once. `%pip` installs into the kernel this notebook is actually running on,
which avoids the common failure where packages are installed into a different Python
than the one Jupyter is using.

In [ ]:
%pip install -q google-api-python-client python-dotenv yt-dlp pandas requests matplotlib

import sys
print("Kernel Python:", sys.executable)

# Verify every import the notebook needs
for m in ["googleapiclient", "dotenv", "yt_dlp", "pandas", "requests", "matplotlib"]:
    try:
        __import__(m)
        print(f"  ok   {m}")
    except ImportError as e:
        print(f"  FAIL {m}  -> {e}")

In [ ]:
import json, time, os, re, pathlib, datetime as dt
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd, requests
from dotenv import load_dotenv
from googleapiclient.discovery import build
from yt_dlp import YoutubeDL

DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)
TODAY = dt.date.today().isoformat()

load_dotenv()                                   # reads .env in this directory
API_KEY = os.getenv("YOUTUBE_API_KEY")
assert API_KEY, "YOUTUBE_API_KEY not found — check that notebook/.env exists and reads YOUTUBE_API_KEY=..."
yt_api = build("youtube", "v3", developerKey=API_KEY, cache_discovery=False)

CHANNELS = {
    "LinusTechTips": "Flagship / long-form reviews",
    "Techquickie":   "Explainers",
    "ShortCircuit":  "Unboxings",
    "TechLinked":    "Tech news",
}

print(f"API key loaded ({API_KEY[:6]}…{API_KEY[-4:]}) · {len(CHANNELS)} channels · {TODAY}")

## 1. Channel metadata and uploads playlist IDs

Quota: `channels.list` costs 1 unit per call.

The `videoCount` returned here is YouTube's own figure and is used later as a
coverage check against what we actually collected.

In [ ]:
quota = 0

def get_channel(handle):
    global quota
    r = yt_api.channels().list(part="snippet,statistics,contentDetails",
                               forHandle=handle).execute()
    quota += 1
    it = r["items"][0]
    return {
        "handle": handle,
        "channel_id": it["id"],
        "title": it["snippet"]["title"],
        "subs": int(it["statistics"].get("subscriberCount", 0)),
        "total_views": int(it["statistics"].get("viewCount", 0)),
        "video_count": int(it["statistics"].get("videoCount", 0)),
        "uploads": it["contentDetails"]["relatedPlaylists"]["uploads"],
    }

chan = {h: get_channel(h) for h in CHANNELS}
chan_df = pd.DataFrame(chan.values())
chan_df["role"] = chan_df.handle.map(CHANNELS)
chan_df.to_csv(DATA / "channels.csv", index=False)

print(chan_df[["title", "role", "subs", "video_count", "total_views"]].to_string(index=False))
print(f"\nQuota used: {quota}")

## 2. Full video catalog — IDs and exact publish timestamps

`playlistItems.list` returns 50 items per call at 1 quota unit each.
Roughly 11,200 videos across the four channels, so about 225 units.

> Pagination has to be sequential — each page needs the previous response's `nextPageToken` —
> but the four channels are independent and run in parallel.

In [ ]:
def list_uploads(handle):
    """Page through a channel's uploads playlist. Returns [(video_id, publishedAt), ...]."""
    cache = DATA / f"api_ids_{handle}.json"
    if cache.exists():
        return json.loads(cache.read_text())

    api = build("youtube", "v3", developerKey=API_KEY, cache_discovery=False)  # per-thread client
    out, token = [], None
    while True:
        r = api.playlistItems().list(part="contentDetails", maxResults=50,
                                     playlistId=chan[handle]["uploads"],
                                     pageToken=token).execute()
        for it in r["items"]:
            cd = it["contentDetails"]
            out.append((cd["videoId"], cd.get("videoPublishedAt")))
        token = r.get("nextPageToken")
        if not token:
            break
    cache.write_text(json.dumps(out))
    return out

t0 = time.time()
ids = {}
with ThreadPoolExecutor(4) as ex:
    futs = {ex.submit(list_uploads, h): h for h in CHANNELS}
    for f in as_completed(futs):
        h = futs[f]; ids[h] = f.result()
        print(f"  {h:<16} {len(ids[h]):>6} videos")

total_ids = sum(len(v) for v in ids.values())
quota += -(-total_ids // 50)
print(f"\n{total_ids:,} videos in {time.time()-t0:.0f}s · quota ≈ {quota}")

## 3. View counts, engagement and duration

`videos.list` accepts 50 IDs per call at 1 quota unit each — another ~225 units.
These batches are independent, so they run in parallel.

In [ ]:
def iso8601_to_sec(s):
    """'PT1H2M3S' -> 3723"""
    if not s:
        return None
    m = re.fullmatch(r"P(?:(\d+)D)?T(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", s)
    if not m:
        return None
    d, h, mi, se = (int(x) if x else 0 for x in m.groups())
    return d*86400 + h*3600 + mi*60 + se

def fetch_stats(batch):
    api = build("youtube", "v3", developerKey=API_KEY, cache_discovery=False)
    r = api.videos().list(part="snippet,statistics,contentDetails",
                          id=",".join(batch), maxResults=50).execute()
    rows = []
    for it in r["items"]:
        st = it.get("statistics", {})
        rows.append({
            "id": it["id"],
            "title": it["snippet"]["title"],
            "channel_id": it["snippet"]["channelId"],
            "published_at": it["snippet"]["publishedAt"],
            "views": int(st["viewCount"]) if "viewCount" in st else None,
            "likes": int(st["likeCount"]) if "likeCount" in st else None,
            "comments": int(st["commentCount"]) if "commentCount" in st else None,
            "duration_sec": iso8601_to_sec(it["contentDetails"].get("duration")),
        })
    return rows

cache = DATA / "api_videos.json"
if cache.exists():
    vids = json.loads(cache.read_text())
    print(f"Loaded {len(vids):,} videos from cache")
else:
    all_ids = [v for lst in ids.values() for v, _ in lst]
    batches = [all_ids[i:i+50] for i in range(0, len(all_ids), 50)]
    print(f"{len(all_ids):,} videos in {len(batches)} batches")
    vids, t0 = [], time.time()
    with ThreadPoolExecutor(8) as ex:
        for i, rows in enumerate(ex.map(fetch_stats, batches), 1):
            vids += rows
            if i % 50 == 0:
                print(f"    batch {i}/{len(batches)}  ({time.time()-t0:.0f}s)")
    cache.write_text(json.dumps(vids))
    quota += len(batches)
    print(f"Done: {len(vids):,} videos in {time.time()-t0:.0f}s · quota ≈ {quota}")

## 4. Shorts classification

The API gives no Shorts flag, so we pull the `/shorts` tab and treat that ID set as
YouTube's own classification. Flat mode is enough here — it omits `duration` and
`timestamp` for Shorts (measured at 0% completeness) but the IDs are complete, and the
IDs are all we need since the API already supplied the rest.

In [ ]:
def shorts_ids(handle):
    cache = DATA / f"shortids_{handle}.txt"
    if cache.exists():
        return [l.strip().split("/")[-1] for l in cache.open() if l.strip()]
    with YoutubeDL({"quiet": True, "skip_download": True,
                    "extract_flat": "in_playlist"}) as ydl:
        info = ydl.extract_info(f"https://www.youtube.com/@{handle}/shorts", download=False)
    out = [r["id"] for r in (info.get("entries") or []) if r]
    cache.write_text("\n".join(out))
    return out

with ThreadPoolExecutor(4) as ex:
    short_sets = dict(zip(CHANNELS, ex.map(shorts_ids, CHANNELS)))
SHORT_IDS = {v for lst in short_sets.values() for v in lst}

for h, lst in short_sets.items():
    print(f"  {h:<16} {len(lst):>4} Shorts")
print(f"\n{len(SHORT_IDS)} Shorts total")

## 5. Build the master table

Every date and view count in this table is exact — no approximation, no rounding.

In [ ]:
id2handle = {c["channel_id"]: c["handle"] for c in chan.values()}

master = pd.DataFrame(vids)
master["channel"] = master.channel_id.map(id2handle)
master["role"] = master.channel.map(CHANNELS)
master["kind"] = master.id.map(lambda i: "short" if i in SHORT_IDS else "long")
master["date"] = pd.to_datetime(master.published_at, errors="coerce", utc=True).dt.tz_localize(None)
master["duration_min"] = master.duration_sec / 60
master = (master.dropna(subset=["channel"])
                .drop_duplicates("id")
                .sort_values("date")
                .reset_index(drop=True))

master.to_csv(DATA / "master_videos.csv", index=False)

print(master.pivot_table(index="channel", columns="kind", values="id",
                         aggfunc="count", fill_value=0).to_string())
print(f"\n{len(master):,} videos · {master.views.sum():,.0f} cumulative views")
print(f"Date range: {master.date.min().date()} → {master.date.max().date()}")

In [ ]:
# Coverage check against YouTube's own videoCount.
# Small shortfalls are expected: private, deleted and members-only uploads are
# counted by the channel statistics but are not retrievable.
cov = (master.groupby("channel").size().rename("collected").to_frame()
         .join(chan_df.set_index("handle").video_count.rename("reported")))
cov["coverage"] = (cov.collected / cov.reported).map("{:.1%}".format)
print(cov.to_string())

## 6. LTT Store

Shopify's `page` parameter is an independent integer rather than a cursor, so pages can be
requested concurrently.

> **Scope note.** This endpoint returns only products **currently on sale**; anything
> delisted is absent. We therefore do **not** build a "product launch cadence" time series —
> it would systematically understate early expansion, because the older a year is, the more
> likely its products have since been delisted. The analysis is limited to the cross-sectional
> structure of the current catalog: category mix, price architecture, SKU complexity,
> stock-out rate.
>
> `published_at` records when a product was published to a sales channel, and it resets if a
> product is delisted and relisted. It is a **lower bound** on first-sale date, kept for
> description only and never used as evidence for a trend or a causal claim.

In [ ]:
def fetch_shopify(domain="www.lttstore.com", max_pages=20, workers=8):
    def one(page):
        r = requests.get(f"https://{domain}/products.json",
                         params={"limit": 250, "page": page}, timeout=30)
        r.raise_for_status()
        return r.json().get("products", [])
    with ThreadPoolExecutor(workers) as ex:
        pages = list(ex.map(one, range(1, max_pages + 1)))
    products = []
    for p in pages:
        if not p:
            break            # first empty page marks the end
        products += p
    return products

products = fetch_shopify()
(DATA / "lttstore_raw.json").write_text(json.dumps(products))

# Explode to variant level — one row per colour/size combination
sku = pd.DataFrame([{
    "product": p["title"], "handle": p["handle"],
    "product_type": p.get("product_type") or "(uncategorised)",
    "published_at": pd.to_datetime(p.get("published_at"), errors="coerce", utc=True),
    "variant": v.get("title"), "sku": v.get("sku"),
    "price": pd.to_numeric(v.get("price"), errors="coerce"),
    "available": v.get("available"),
} for p in products for v in p.get("variants", [])])

sku.to_csv(DATA / f"lttstore_skus_{TODAY}.csv", index=False)

print(f"{sku.handle.nunique()} products → {len(sku)} variants "
      f"({len(sku)/sku.handle.nunique():.1f} SKUs per product)")
print(f"Out of stock: {(~sku.available).mean():.1%} of variants")
print(f"Price range: ${sku.price.min():.2f} – ${sku.price.max():.2f} "
      f"(median ${sku.price.median():.2f})")
sku.head()

In [ ]:
print("=== Collection complete ===")
print(f"Videos          {len(master):,}")
print(f"Store variants  {len(sku):,}")
print(f"API quota used  ~{quota} of 10,000 daily")
print(f"\nWritten to {DATA.resolve()}")
for f in sorted(DATA.iterdir()):
    print(f"  {f.name:<34} {f.stat().st_size/1024:>9,.0f} KB")

---

## Known limitations

Carried into the analysis and stated explicitly in the presentation.

1. **View counts are cumulative, not a time series.** A 2013 video has been accumulating for
   thirteen years; last week's has not. Cross-period comparison of absolute view counts is
   therefore biased toward older videos. Attempting to estimate the accrual curve from
   cross-sectional data fails because age and era are perfectly collinear — a decline in recent
   per-video performance is indistinguishable from views not yet accrued. The analysis instead
   uses a **rolling baseline index**: each video divided by the median of the nine that preceded
   it on the same channel, which holds era approximately constant by construction.
2. **No sales data.** `products.json` carries price and availability but no units sold, so
   revenue cannot be derived — only price architecture and assortment.
3. **Floatplane scale is not observable.** Pricing is public; subscriber counts are not.
4. **Coverage shortfall.** Private, deleted and members-only uploads are included in YouTube's
   reported `videoCount` but cannot be retrieved.

Re-running this notebook reads from `data/` and consumes no additional quota.
To force a refresh, delete the relevant `api_*.json` files.

Next: `02_analysis.ipynb`.